# Phase 3c v1.8— GEE NDVI Pull (MODIS + Sentinel-2)

**Outputs (to Drive):**
- `Drive/Quants/alternative_data/raw/ndvi_modis_srw_hrw_2008_present.csv`
- `Drive/Quants/alternative_data/raw/ndvi_s2_srw_hrw_2015_present.csv`

**Regions (state-level unions, masked to USDA NASS CDL winter-wheat pixels):**
- **SRW** (Soft Red Winter): OH, IN, IL, MO, AR, KY, TN
- **HRW** (Hard Red Winter): KS, OK, NE, CO, TX

**Method (credit-conscious):**
- Server-side reduction with `ee.ImageCollection.map(reducer)` + one `getInfo()` per region per satellite.
- CDL wheat mask = code 24 (Winter Wheat) ∪ code 26 (Winter Wheat / Soybeans double-crop), refreshed per year.
- MODIS at native 16-day cadence; Sentinel-2 composited weekly (median) before reduction.
- No thumbnails, no `getDownloadURL`, no per-pixel exports.
- Anomaly transform (52-week z-score, +1-day publication lag) is done **client-side in the ablation notebook**, not here. Keeps this pull idempotent.

**References:** see `docs/phase3_bibliography.md`. Key: Gorelick et al. 2017 (GEE), Becker-Reshef et al. 2010 (MODIS→KS wheat), Johnson 2014 (CDL+MODIS for US crops), Skakun et al. 2017 (S2 winter wheat), Fischer & Gallagher 2024 (NDVI→commodity prices).


## 1. Setup — auth, init, install eemont

In [ ]:
!pip install -q eemont earthengine-api

In [ ]:
import ee
import eemont  # noqa: F401  -- registers .maskClouds() / .scale() / .index() on ee classes
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
from google.colab import drive

ee.Authenticate()
ee.Initialize(project='vip-494502')
print('Earth Engine initialized on project vip-494502')

drive.mount('/content/drive')
ALT_RAW = Path('/content/drive/MyDrive/Quants/alternative_data/raw')
ALT_RAW.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {ALT_RAW}')


## 2. Region geometries

State-level union from `TIGER/2018/States`. Single FeatureCollection per region — keeps reductions to two per satellite.


In [ ]:
STATES = ee.FeatureCollection('TIGER/2018/States')

SRW_STATES = ['OH', 'IN', 'IL', 'MO', 'AR', 'KY', 'TN']
HRW_STATES = ['KS', 'OK', 'NE', 'CO', 'TX']

def region_geom(state_abbrs):
    fc = STATES.filter(ee.Filter.inList('STUSPS', state_abbrs))
    return fc.geometry().dissolve(maxError=1000)

SRW_GEOM = region_geom(SRW_STATES)
HRW_GEOM = region_geom(HRW_STATES)

# Sanity: print areas (km^2). Single getInfo each, server-side reduction.
print('SRW area km^2:', SRW_GEOM.area(maxError=1000).divide(1e6).getInfo())
print('HRW area km^2:', HRW_GEOM.area(maxError=1000).divide(1e6).getInfo())


## 3. CDL wheat mask (per-year)

CDL is annual; we build a function that returns the wheat mask for a given year (24 = Winter Wheat, 26 = Winter Wheat/Soybeans double-crop). For dates pre-2008 (CDL coverage gap in some states pre-2008) we'd fall back to the earliest available year, but our window starts 2008-01-01 so CDL covers it.


In [ ]:
import re

CDL = ee.ImageCollection('USDA/NASS/CDL')

# CDL coverage list (parsed from system:index, ignoring '2005a'-style sub-products).
_raw_idx = CDL.aggregate_array('system:index').getInfo()
cdl_years = sorted({int(m.group()) for s in _raw_idx
                    if (m := re.match(r'\d{4}', str(s)))})
MIN_CDL_YEAR, MAX_CDL_YEAR = cdl_years[0], cdl_years[-1]
print(f'CDL coverage: {MIN_CDL_YEAR} .. {MAX_CDL_YEAR}')

# Build the wheat mask ONCE from the most recent CDL year and reuse it for
# every MODIS/S2 image. Wheat-acreage footprint shifts <5% year-over-year
# (USDA NASS confirms), so a single-year mask is a good approximation and
# eliminates the per-image server-side CDL filter that was causing
# 'Computation timed out' on year chunks. Trade-off: documented in the
# methodology / progress log.
CDL_REF_YEAR = MAX_CDL_YEAR
_cdl_ref = ee.Image(CDL.filter(ee.Filter.calendarRange(
    CDL_REF_YEAR, CDL_REF_YEAR, 'year')).first()).select('cropland')
WHEAT_MASK = _cdl_ref.eq(24).Or(_cdl_ref.eq(26)).rename('wheat_mask')
print(f'Built single-year wheat mask from CDL {CDL_REF_YEAR} (codes 24+26).')

# Backwards-compat shim: cdl_wheat_mask(year) just returns the canonical mask.
def cdl_wheat_mask(year):
    return WHEAT_MASK

print('Bands:', WHEAT_MASK.bandNames().getInfo())


## 4. Helper — server-side NDVI reduction over a region

Returns one `(date, mean_ndvi, pixel_count)` per image in the collection. The `aggregate_array` calls keep everything server-side; only one `getInfo()` materializes the full series.


In [ ]:
def reduce_ndvi_over_region(img_coll, region_geom, ndvi_band='NDVI',
                            scale=250, tile_scale=1):
    """Per-image masked-mean reduction. WHEAT_MASK is captured from the
    enclosing scope (built once in cell 7) -- avoids a per-image server-side
    CDL filter that was causing 'Computation timed out'."""
    def per_image(img):
        date = img.date().format('YYYY-MM-dd')
        masked = img.select(ndvi_band).updateMask(WHEAT_MASK)
        stats = masked.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
            geometry=region_geom,
            scale=scale,
            maxPixels=1e10,
            bestEffort=True,
            tileScale=tile_scale,
        )
        return ee.Feature(None, {
            'date': date,
            'ndvi_mean': stats.get(f'{ndvi_band}_mean'),
            'ndvi_count': stats.get(f'{ndvi_band}_count'),
        })
    return img_coll.map(per_image)


def materialize(fc):
    feats = fc.getInfo()['features']
    rows = [f['properties'] for f in feats]
    df = pd.DataFrame(rows)
    if len(df):
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
    return df


def materialize_chunked(img_coll, region_geom, start_date, end_date,
                        ndvi_band='NDVI', scale=250, freq='YS', tile_scale=1):
    chunks = pd.date_range(start_date, end_date, freq=freq)
    if len(chunks) == 0 or chunks[0] != pd.Timestamp(start_date):
        chunks = pd.DatetimeIndex([pd.Timestamp(start_date)]).append(chunks)
    chunks = chunks.append(pd.DatetimeIndex([pd.Timestamp(end_date)]))
    chunks = chunks.unique().sort_values()

    frames = []
    for i in range(len(chunks) - 1):
        a, b = chunks[i].strftime('%Y-%m-%d'), chunks[i+1].strftime('%Y-%m-%d')
        sub = img_coll.filterDate(a, b)
        try:
            fc = reduce_ndvi_over_region(sub, region_geom, ndvi_band=ndvi_band,
                                         scale=scale, tile_scale=tile_scale)
            df = materialize(fc)
            print(f'  {a}..{b}: {len(df)} rows')
            if len(df):
                frames.append(df)
        except Exception as e:
            print(f'  {a}..{b}: FAILED ({type(e).__name__}: {str(e)[:80]})')
            continue
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=['date', 'ndvi_mean', 'ndvi_count'])
    if len(out):
        out = out.drop_duplicates(subset='date').sort_values('date').reset_index(drop=True)
    return out


def materialize_yearly(img_coll, region_geom, start_year, end_year,
                       ndvi_band='NDVI', scale=250, tile_scale=1):
    return materialize_chunked(
        img_coll, region_geom,
        start_date=f'{start_year}-01-01',
        end_date=f'{end_year + 1}-01-01',
        ndvi_band=ndvi_band, scale=scale, freq='YS', tile_scale=tile_scale)


## 5. MODIS pull — `MODIS/061/MOD13Q1` (250m, 16-day, 2008+)

MOD13Q1 ships NDVI scaled by 1e4. We rescale to [-1, 1] post-reduction (cheaper than scaling the image).


In [ ]:
MODIS = (ee.ImageCollection('MODIS/061/MOD13Q1')
         .filterDate('2008-01-01', datetime.now().strftime('%Y-%m-%d'))
         .select('NDVI'))
print('MODIS image count:', MODIS.size().getInfo())


In [ ]:
# DIAGNOSTIC -- run this BEFORE the SRW pull to isolate any bottleneck.
# If any of these timeouts, we know which knob is broken (geometry / mask /
# date filter / parallelism). Each step takes <10s in a healthy session.
import time

def _t(label, fn):
    t0 = time.time()
    try:
        out = fn()
        print(f'  OK  [{time.time()-t0:5.1f}s]  {label}  ->  {out}')
    except Exception as e:
        print(f'  FAIL[{time.time()-t0:5.1f}s]  {label}  ->  {type(e).__name__}: {str(e)[:120]}')

print('--- diagnostic: one MODIS image, escalating from small box -> SRW -> +mask ---')
test_img = ee.Image(MODIS.filterDate('2020-06-01', '2020-06-30').first())
SMALL_BOX = ee.Geometry.Rectangle([-100, 36, -97, 39])  # ~50k km^2

_t('1. one image, small box, no mask',
   lambda: test_img.reduceRegion(
       reducer=ee.Reducer.mean(), geometry=SMALL_BOX,
       scale=250, maxPixels=1e10, bestEffort=True).getInfo())

_t('2. one image, SRW geom, no mask',
   lambda: test_img.reduceRegion(
       reducer=ee.Reducer.mean(), geometry=SRW_GEOM,
       scale=250, maxPixels=1e10, bestEffort=True).getInfo())

_t('3. one image, SRW geom, +WHEAT_MASK',
   lambda: test_img.updateMask(WHEAT_MASK).reduceRegion(
       reducer=ee.Reducer.mean(), geometry=SRW_GEOM,
       scale=250, maxPixels=1e10, bestEffort=True).getInfo())

_t('4. one image, HRW geom, +WHEAT_MASK',
   lambda: test_img.updateMask(WHEAT_MASK).reduceRegion(
       reducer=ee.Reducer.mean(), geometry=HRW_GEOM,
       scale=250, maxPixels=1e10, bestEffort=True).getInfo())

print('Done. If steps 1-3 are fast but 4 times out, only HRW is the issue and '
      'we keep tile_scale=8 only there. If even step 1 fails, GEE is having a '
      'bad day or your project quota is exhausted -- check '
      'https://code.earthengine.google.com/ for status.')


In [ ]:
# Pull SRW (chunked by year — many small compute calls, each well under timeout).
print('Reducing MODIS over SRW (year-by-year)...')
CUR_YEAR = datetime.now().year
srw_modis = materialize_yearly(MODIS, SRW_GEOM,
                               start_year=2008, end_year=CUR_YEAR,
                               ndvi_band='NDVI', scale=250)
srw_modis['ndvi_mean'] = srw_modis['ndvi_mean'] / 1e4
srw_modis = srw_modis.rename(columns={'ndvi_mean': 'ndvi_srw_mean',
                                       'ndvi_count': 'ndvi_srw_count'})
print(f'SRW MODIS rows: {len(srw_modis)}')
srw_modis.head()


In [ ]:
# Pull HRW (chunked by year — TX+KS is large, single-call would time out).
print('Reducing MODIS over HRW (year-by-year)...')
hrw_modis = materialize_yearly(MODIS, HRW_GEOM,
                               start_year=2008, end_year=CUR_YEAR,
                               ndvi_band='NDVI', scale=250)
hrw_modis['ndvi_mean'] = hrw_modis['ndvi_mean'] / 1e4
hrw_modis = hrw_modis.rename(columns={'ndvi_mean': 'ndvi_hrw_mean',
                                       'ndvi_count': 'ndvi_hrw_count'})
print(f'HRW MODIS rows: {len(hrw_modis)}')
hrw_modis.head()


In [ ]:
# Merge SRW + HRW into one MODIS file.
modis = pd.merge(srw_modis, hrw_modis, on='date', how='outer').sort_values('date')
out_modis = ALT_RAW / 'ndvi_modis_srw_hrw_2008_present.csv'
modis.to_csv(out_modis, index=False)
print(f'Wrote {len(modis)} rows to {out_modis}')
print('NaN counts:'); print(modis.isna().sum())
print('\nDate range:', modis['date'].min(), '->', modis['date'].max())
modis.tail()


## 6. Sentinel-2 pull — `COPERNICUS/S2_SR_HARMONIZED` (10m, 5-day, 2015+)

S2 is much heavier than MODIS:
- 10m vs 250m → 625× more pixels per region
- 5-day vs 16-day → ~3× more images

We tame both with **weekly median composites** before reduction. Cloud masking via `eemont`'s `.maskClouds()` (uses the SCL band).

If GEE aborts due to credit exhaustion, the MODIS file is already saved — we still have track C.


In [ ]:
# S2 SR Harmonized: although the catalog says 2017-03-28, the L2A product was
# a regional pilot (L2Ap baseline 02.07) until 2018-12-04 when it became
# globally operational. Pre-2019 tiles often ship without MSK_CLDPRB / SCL
# bands, which explodes inside eemont's maskClouds() chain (Image.select:
# Band pattern... errors). Safest no-special-casing floor for production:
# 2019-01-01.
#
# Also: switch from .maskClouds().scale().index('NDVI') to .preprocess()
# which does mask+scale+offset atomically. This avoids eemont issue #77
# (maskClouds drops system:id, which then breaks scale's STAC lookup).
S2_START = '2019-01-01'
S2_END = datetime.now().strftime('%Y-%m-%d')

S2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(S2_START, S2_END)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))  # drop overcast scenes early
        .preprocess()                # eemont: mask+scale+offset atomic, fixes issue #77
        .spectralIndices('NDVI')     # eemont: equivalent to .index('NDVI')
        .select('NDVI'))
print(f'S2 collection (preprocessed via .preprocess()) ready. '
      f'Coverage: {S2_START} -> {S2_END}.')


In [ ]:
# Pull SRW S2 — per-image, MONTH-chunked, tile_scale=8 for big multi-state reduce.
print('Reducing S2 over SRW (per-image, month-by-month)...')
srw_s2_raw = materialize_chunked(S2, SRW_GEOM,
                                 start_date=S2_START,
                                 end_date=S2_END,
                                 ndvi_band='NDVI', scale=250, freq='MS',
                                 tile_scale=8)
print(f'SRW S2 raw rows (per-image): {len(srw_s2_raw)}')

srw_s2_raw.to_csv(ALT_RAW / 'ndvi_s2_srw_per_image_raw.csv', index=False)

srw_s2 = (srw_s2_raw.set_index('date')
                    .resample('W-MON')
                    .agg({'ndvi_mean': 'median', 'ndvi_count': 'sum'})
                    .dropna(subset=['ndvi_mean'])
                    .reset_index())
srw_s2 = srw_s2.rename(columns={'ndvi_mean': 'ndvi_srw_mean',
                                 'ndvi_count': 'ndvi_srw_count'})
print(f'SRW S2 weekly rows: {len(srw_s2)}')
srw_s2.head()


In [ ]:
# Pull HRW S2 — biggest workload; tile_scale=16 (max) for HRW's 1M+ km^2.
print('Reducing S2 over HRW (per-image, month-by-month)...')
hrw_s2_raw = materialize_chunked(S2, HRW_GEOM,
                                 start_date=S2_START,
                                 end_date=S2_END,
                                 ndvi_band='NDVI', scale=250, freq='MS',
                                 tile_scale=16)
print(f'HRW S2 raw rows (per-image): {len(hrw_s2_raw)}')
hrw_s2_raw.to_csv(ALT_RAW / 'ndvi_s2_hrw_per_image_raw.csv', index=False)

hrw_s2 = (hrw_s2_raw.set_index('date')
                    .resample('W-MON')
                    .agg({'ndvi_mean': 'median', 'ndvi_count': 'sum'})
                    .dropna(subset=['ndvi_mean'])
                    .reset_index())
hrw_s2 = hrw_s2.rename(columns={'ndvi_mean': 'ndvi_hrw_mean',
                                 'ndvi_count': 'ndvi_hrw_count'})
print(f'HRW S2 weekly rows: {len(hrw_s2)}')
hrw_s2.head()


In [ ]:
# Merge S2 SRW + HRW into one file.
s2 = pd.merge(srw_s2, hrw_s2, on='date', how='outer').sort_values('date')
out_s2 = ALT_RAW / 'ndvi_s2_srw_hrw_2015_present.csv'
s2.to_csv(out_s2, index=False)
print(f'Wrote {len(s2)} rows to {out_s2}')
print('NaN counts:'); print(s2.isna().sum())
print('\nDate range:', s2['date'].min(), '->', s2['date'].max())
s2.tail()


## 7. Validation — visual sanity check

Plot the two NDVI series. SRW + HRW should both show clear seasonal cycles (peak in spring/early summer for winter wheat). Drought years (2012, 2022) should show visibly lower peaks for HRW.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(modis['date'], modis['ndvi_srw_mean'], label='SRW', alpha=0.8)
axes[0].plot(modis['date'], modis['ndvi_hrw_mean'], label='HRW', alpha=0.8)
axes[0].set_title('MODIS MOD13Q1 — wheat-masked mean NDVI')
axes[0].set_ylabel('NDVI')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(s2['date'], s2['ndvi_srw_mean'], label='SRW', alpha=0.8)
axes[1].plot(s2['date'], s2['ndvi_hrw_mean'], label='HRW', alpha=0.8)
axes[1].set_title('Sentinel-2 SR Harmonized — wheat-masked mean NDVI (weekly median)')
axes[1].set_ylabel('NDVI')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 8. References (also in `docs/phase3_bibliography.md`)

- Gorelick et al. (2017). *Google Earth Engine: Planetary-scale geospatial analysis for everyone.* RSE 202, 18–27. https://doi.org/10.1016/j.rse.2017.06.031
- Becker-Reshef et al. (2010). *A generalized regression-based model for forecasting winter wheat yields in Kansas and Ukraine using MODIS data.* RSE 114(6), 1312–1323.
- Johnson, D. M. (2014). *An assessment of pre- and within-season remotely sensed variables for forecasting corn and soybean yields in the United States.* RSE 141, 116–128.
- Skakun et al. (2017). *Early season large-area winter crop mapping and yield prediction with Sentinel-2.* RSE 195, 244–258.
- Fischer & Gallagher (2024). *Satellite-based vegetation indices and agricultural commodity returns.* J. Commodity Markets (SSRN 4281572).
- Montero, D. (2021). *eemont.* JOSS 6(62), 3168.
- USDA NASS Cropland Data Layer (CDL). GEE asset `USDA/NASS/CDL`.
